In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [9]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [10]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Membaca file CSV ke dalam Spark DataFrame
df_tugas = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

# Mengubah kolom kategorikal menjadi indeks numerik
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx").fit(df_tugas)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_tugas)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx").fit(df_tugas)

df_tugas_idx = indexer_menu.transform(df_tugas)
df_tugas_idx = indexer_cabang.transform(df_tugas_idx)
df_tugas_idx = indexer_waktu.transform(df_tugas_idx)

# Menggabungkan seluruh fitur ke dalam satu kolom vector "features"
assembler_tugas = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

df_tugas_final = assembler_tugas.transform(df_tugas_idx)
df_tugas_final.select("features", "total_penjualan").show(5, truncate=False)

[Stage 11:>                                                         (0 + 1) / 1]

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [11]:
from pyspark.ml.regression import LinearRegression

# Membagi data 75% training dan 25% testing (seed=101)
train_tugas, test_tugas = df_tugas_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train_tugas.count()}")
print(f"Jumlah data testing  : {test_tugas.count()}")

# Melatih model LinearRegression
lr_tugas = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_tugas = lr_tugas.fit(train_tugas)

print("Koefisien:", model_tugas.coefficients)
print("Intercept:", model_tugas.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661


Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


In [12]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions_tugas = model_tugas.transform(test_tugas)

evaluator_rmse_t = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2_t = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_tugas = evaluator_rmse_t.evaluate(predictions_tugas)
r2_tugas = evaluator_r2_t.evaluate(predictions_tugas)

print(f"RMSE Model Utama (5 fitur) : Rp {rmse_tugas:,.0f}")
print(f"R² Model Utama (5 fitur)   : {r2_tugas:.4f}")

RMSE Model Utama (5 fitur) : Rp 10,901
R² Model Utama (5 fitur)   : 0.9107


In [13]:
# Membuat VectorAssembler kedua hanya menggunakan fitur qty dan harga_satuan
assembler_2fitur = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)

df_tugas_2fitur = assembler_2fitur.transform(df_tugas)
train_2f, test_2f = df_tugas_2fitur.randomSplit([0.75, 0.25], seed=101)

# Melatih dan mengevaluasi model kedua
lr_2fitur = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_2fitur = lr_2fitur.fit(train_2f)
pred_2fitur = model_2fitur.transform(test_2f)

rmse_2fitur = evaluator_rmse_t.evaluate(pred_2fitur)
r2_2fitur = evaluator_r2_t.evaluate(pred_2fitur)

print(f"RMSE Model Kedua (2 fitur) : Rp {rmse_2fitur:,.0f}")
print(f"R² Model Kedua (2 fitur)   : {r2_2fitur:.4f}")

RMSE Model Kedua (2 fitur) : Rp 10,896
R² Model Kedua (2 fitur)   : 0.9108


In [14]:
# Membuat DataFrame baru berisi 3 skenario pesanan dummy
data_skenario = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Cappuccino", "Yogyakarta", "Sore", 2, 25000),
    ("Roti Bakar", "Semarang", "Malam", 4, 18000)
]
kolom_skenario = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_skenario = spark.createDataFrame(data_skenario, kolom_skenario)

# Transformasi menggunakan model StringIndexer & VectorAssembler yang SUDAH di-fit sebelumnya
df_skenario_idx = indexer_menu.transform(df_skenario)
df_skenario_idx = indexer_cabang.transform(df_skenario_idx)
df_skenario_idx = indexer_waktu.transform(df_skenario_idx)
df_skenario_final = assembler_tugas.transform(df_skenario_idx)

# Melakukan prediksi menggunakan model utama
prediksi_skenario = model_tugas.transform(df_skenario_final)
prediksi_skenario.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |3  |20000       |59572.01525544525|
|Cappuccino|Yogyakarta|Sore       |2  |25000       |53814.94297982256|
|Roti Bakar|Semarang  |Malam      |4  |18000       |76101.40186674664|
+----------+----------+-----------+---+------------+-----------------+



In [15]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
